# LSTM — Búsqueda de Hiperparámetros con Optuna (L1, L2, L3)

**Objetivo:** entrenar un modelo LSTM para predicción de deterioro clínico en urgencias usando secuencias temporales de constantes vitales + features estáticas del triage.

**Framework:** TensorFlow/Keras — Functional API para combinar la rama secuencial (LSTM) con la rama estática (features del triage).

**Features:** secuencias de 6 vitales × max_len=10 + 10 features estáticas (6 vitales triage + acuity + n_medications + pain + chiefcomplaint codificado) + OHE de arrival_transport.

**Estrategia:** Optuna `TPESampler(seed=42)`, **30 trials × 3 targets**, maximizando AUROC val. Reentrenamiento final con max_epochs=30, patience=7.

## 1. Setup

Importaciones, configuración de semillas de reproducibilidad y constantes globales. `N_TRIALS=30` asigna al LSTM el mismo presupuesto que CatBoost y LogReg.

In [1]:
import os, json, pickle, time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, LSTM, Dense, Dropout,
                                      Concatenate, Masking, BatchNormalization)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from dotenv import load_dotenv
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Suprimir logs de TF y Optuna
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Seeds de reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

load_dotenv(dotenv_path=Path('../../.env'), override=True)
if not os.getenv('MIMIC_IV_ED_PATH'):
    load_dotenv(dotenv_path=Path('.env'), override=True)

DATA          = Path(os.getenv('MIMIC_IV_ED_PATH', ''))
PROCESSED_DIR = Path('../../data/processed')

N_TRIALS = 30
TARGETS = ['L1', 'L2', 'L3']
TARGET_NAMES = {'L1': 'Ingreso hospitalario', 'L2': 'Resultado crítico', 'L3': 'Intervención crítica'}

print(f'TensorFlow {tf.__version__} | GPU disponible: {len(tf.config.list_physical_devices("GPU")) > 0}')
print(f'Optuna {optuna.__version__} — N_TRIALS={N_TRIALS} por target')
print(f'DATA: {DATA}')
print(f'PROCESSED_DIR: {PROCESSED_DIR.resolve()}')

TensorFlow 2.21.0 | GPU disponible: False
Optuna 4.8.0 — N_TRIALS=30 por target
DATA: C:\Users\cuent\Documents\UAX\TFM\TFM\CodigoGit\data
PROCESSED_DIR: C:\Users\cuent\Documents\UAX\TFM\TFM\CodigoGit\data\processed


## 2. Carga de secuencias y features estáticas

Se cargan las secuencias NPZ y los parquets de train/val. Las secuencias deben estar alineadas por posición con las filas del parquet; se verifican los asserts de alineación al final de la celda de carga.

In [2]:
data_train = np.load(PROCESSED_DIR / 'lstm_sequences_train.npz')
data_val   = np.load(PROCESSED_DIR / 'lstm_sequences_val.npz')
X_seq_train = data_train['X_seq']
seq_len_train = data_train['seq_len']
X_seq_val = data_val['X_seq']
seq_len_val = data_val['seq_len']

with open(PROCESSED_DIR / 'lstm_stats.json') as f:
    lstm_stats = json.load(f)
MAX_LEN = lstm_stats['max_len']
print(f'X_seq_train: {X_seq_train.shape}  |  X_seq_val: {X_seq_val.shape}  |  MAX_LEN={MAX_LEN}')

df_train = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
df_val   = pd.read_parquet(PROCESSED_DIR / 'val.parquet')

df_medrecon = pd.read_csv(DATA / 'medrecon.csv', low_memory=False, usecols=['stay_id'])
n_meds = df_medrecon.groupby('stay_id').size().rename('n_medications').reset_index()
for df in [df_train, df_val]:
    df.drop(columns=['n_medications'], errors='ignore', inplace=True)
df_train = df_train.merge(n_meds, on='stay_id', how='left')
df_val   = df_val.merge(n_meds, on='stay_id', how='left')
df_train['n_medications'] = df_train['n_medications'].fillna(0).astype(int)
df_val['n_medications']   = df_val['n_medications'].fillna(0).astype(int)

# pain se almacena como str en los parquets; coerción a float antes de imputación
df_train['pain'] = pd.to_numeric(df_train['pain'], errors='coerce')
df_val['pain']   = pd.to_numeric(df_val['pain'],   errors='coerce')

# ── Codificación de chiefcomplaint (vocabulario construido solo sobre train) ──
CC_TOP_N = 200
cc_counts = df_train['chiefcomplaint'].fillna('').str.lower().str.strip().value_counts()
cc_vocab  = {cc: i + 1 for i, cc in enumerate(cc_counts.head(CC_TOP_N).index.tolist())}
CC_OOV    = CC_TOP_N + 1  # índice para valores fuera de vocabulario

def encode_cc(series, vocab, oov_idx):
    out = []
    for v in series:
        if pd.isna(v) or str(v).strip() == '':
            out.append(0.0)  # 0 = ausente/NaN
        else:
            out.append(float(vocab.get(str(v).lower().strip(), oov_idx)))
    return np.array(out, dtype=np.float32)

df_train['cc_encoded'] = encode_cc(df_train['chiefcomplaint'], cc_vocab, CC_OOV)
df_val['cc_encoded']   = encode_cc(df_val['chiefcomplaint'],   cc_vocab, CC_OOV)

# ── Features estáticas: 9 originales + chiefcomplaint codificado ──
STATIC_FEATURES = ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp',
                   'acuity', 'n_medications', 'pain', 'cc_encoded']
imputer_s = SimpleImputer(strategy='median')
scaler_s  = StandardScaler()
X_static_train = scaler_s.fit_transform(imputer_s.fit_transform(df_train[STATIC_FEATURES])).astype(np.float32)
X_static_val   = scaler_s.transform(imputer_s.transform(df_val[STATIC_FEATURES])).astype(np.float32)

# ── One-hot encoding de arrival_transport (ajustado sobre train) ──
ohe_s = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_transport_train = ohe_s.fit_transform(df_train[['arrival_transport']].fillna('desconocido'))
X_transport_val   = ohe_s.transform(df_val[['arrival_transport']].fillna('desconocido'))
X_static_train = np.hstack([X_static_train, X_transport_train]).astype(np.float32)
X_static_val   = np.hstack([X_static_val,   X_transport_val  ]).astype(np.float32)

assert len(X_seq_train) == len(X_static_train) == len(df_train)
assert len(X_seq_val)   == len(X_static_val)   == len(df_val)
print(f'X_static_train: {X_static_train.shape}  |  X_static_val: {X_static_val.shape}')
print(f'  → {len(STATIC_FEATURES)} numéricas + {X_transport_train.shape[1]} OHE (arrival_transport)')
print(f'Vocabulario chiefcomplaint: {len(cc_vocab)} términos + OOV + ausente')

X_seq_train: (278320, 10, 6)  |  X_seq_val: (59640, 10, 6)  |  MAX_LEN=10
X_static_train: (278320, 15)  |  X_static_val: (59640, 15)
  → 10 numéricas + 5 OHE (arrival_transport)
Vocabulario chiefcomplaint: 200 términos + OOV + ausente


## 3. Arquitectura LSTM (Keras Functional API)

`build_model` construye el modelo con dos ramas que se fusionan antes de la clasificación final:

- **Rama secuencial:** `Masking` → una o más capas `LSTM` apiladas
- **Rama estática:** `BatchNormalization` sobre las features del triage
- **Fusión:** `Concatenate` → `Dropout` → `Dense(relu)` → `Dense(sigmoid)`

`Masking(mask_value=0.0)` indica a Keras que ignore los pasos de tiempo rellenos con ceros (pacientes con menos de `MAX_LEN` mediciones), lo que equivale al `pack_padded_sequence` estándar de PyTorch.

In [3]:
def build_model(n_vitals, n_static, units, n_layers, dropout, fc_hidden, lr):
    """Construye el modelo LSTM con Keras Functional API."""
    # --- Rama secuencial ---
    input_seq = Input(shape=(MAX_LEN, n_vitals), name='seq')
    x = Masking(mask_value=0.0)(input_seq)          # ignora pasos de relleno
    for i in range(n_layers):
        x = LSTM(units, return_sequences=(i < n_layers - 1), dropout=dropout)(x)

    # --- Rama estática ---
    input_static = Input(shape=(n_static,), name='static')
    s = BatchNormalization()(input_static)

    # --- Fusión y clasificación ---
    combined = Concatenate()([x, s])
    combined = Dropout(dropout)(combined)
    combined = Dense(fc_hidden, activation='relu')(combined)
    combined = Dropout(dropout)(combined)
    output   = Dense(1, activation='sigmoid')(combined)

    model = Model(inputs=[input_seq, input_static], outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc')],
    )
    return model

# Verificación rápida de la arquitectura con los hiperparámetros por defecto
_m = build_model(X_seq_train.shape[2], X_static_train.shape[1],
                 units=64, n_layers=1, dropout=0.2, fc_hidden=32, lr=1e-3)
_m.summary()
del _m

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ seq (InputLayer)    │ (None, 10, 6)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 10, 6)     │          0 │ seq[0][0]         │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking (Masking)   │ (None, 10, 6)     │          0 │ seq[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any (Any)           │ (None, 10)        │          0 │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ static (InputLayer) │ (None, 15)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 64)        │     18,176 │ masking[0][0],    │
│                     │                   │            │ any[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 15)        │         60 │ static[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 79)        │          0 │ lstm[0][0],       │
│ (Concatenate)       │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 79)        │          0 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      2,560 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         33 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 20,829 (81.36 KB)

 Trainable params: 20,799 (81.25 KB)

 Non-trainable params: 30 (120.00 B)

## 4. Espacio de búsqueda Optuna

| Parámetro | Rango |
|-----------|-------|
| `units` | {32, 64, 128, 256} |
| `n_layers` | {1, 2, 3} |
| `dropout` | [0.0, 0.5] |
| `fc_hidden` | {16, 32, 64} |
| `lr` | [1e-4, 5e-3] log |
| `batch_size` | {512, 1024, 2048} |

`max_epochs=10` en trials con `EarlyStopping(patience=3)`. Reentrenamiento final con `max_epochs=30, patience=7` y `ReduceLROnPlateau`. `tf.keras.backend.clear_session()` al final de cada trial para liberar memoria de GPU.

In [4]:
def make_objective(y_tr, y_v):
    def objective(trial):
        units      = trial.suggest_categorical('units',      [32, 64, 128, 256])
        n_layers   = trial.suggest_categorical('n_layers',   [1, 2, 3])
        dropout    = trial.suggest_float('dropout',          0.0, 0.5)
        fc_hidden  = trial.suggest_categorical('fc_hidden',  [16, 32, 64])
        lr         = trial.suggest_float('lr',               1e-4, 5e-3, log=True)
        batch_size = trial.suggest_categorical('batch_size', [512, 1024, 2048])

        pos_weight = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))
        model = build_model(
            n_vitals=X_seq_train.shape[2], n_static=X_static_train.shape[1],
            units=units, n_layers=n_layers, dropout=dropout,
            fc_hidden=fc_hidden, lr=lr,
        )
        model.fit(
            [X_seq_train, X_static_train], y_tr,
            validation_data=([X_seq_val, X_static_val], y_v),
            epochs=10, batch_size=batch_size,
            class_weight={0: 1.0, 1: pos_weight},
            callbacks=[EarlyStopping(monitor='val_auc', patience=3,
                                     mode='max', restore_best_weights=True)],
            verbose=0,
        )
        y_pred = model.predict([X_seq_val, X_static_val], verbose=0).ravel()
        tf.keras.backend.clear_session()   # libera memoria GPU entre trials
        return roc_auc_score(y_v, y_pred)
    return objective

## 5. Optimización por target

Se ejecuta la búsqueda bayesiana para cada target. Se registran el número de trials completados y podados, y el tiempo por target, para documentar el presupuesto computacional real.

In [5]:
import time

best_params = {}
best_aurocs = {}
trial_times = {}

t_total_start = time.time()

for target in TARGETS:
    y_tr = df_train[target].values.astype(np.float32)
    y_v  = df_val[target].values.astype(np.float32)

    print(f"\n{'='*60}")
    print(f'Optimizando TARGET: {target} — {TARGET_NAMES[target]}')
    print(f"{'='*60}")

    t_start = time.time()

    sampler = optuna.samplers.TPESampler(seed=SEED)
    pruner  = optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5)
    study   = optuna.create_study(
        direction='maximize', study_name=f'lstm_{target}',
        sampler=sampler, pruner=pruner,
    )
    study.optimize(make_objective(y_tr, y_v), n_trials=N_TRIALS, show_progress_bar=True)

    elapsed = time.time() - t_start
    trial_times[target] = elapsed

    best_params[target] = study.best_params
    best_aurocs[target] = study.best_value
    n_pruned   = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
    n_complete = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
    print(f'\n{target} — Mejor AUROC (Optuna, max_epochs=10, {N_TRIALS} trials): {study.best_value:.4f}')
    print(f'   Trials completos: {n_complete} | pruned: {n_pruned}')
    print(f'   Tiempo: {elapsed:.1f}s ({elapsed/60:.1f} min) | Media por trial: {elapsed/N_TRIALS:.1f}s')
    print(f'   Mejores parámetros: {study.best_params}')

t_total_optuna = time.time() - t_total_start
print(f"\n{'='*60}")
print(f'Búsqueda completada para todos los targets.')
print(f'   Tiempo TOTAL Optuna (3 targets x {N_TRIALS} trials): {t_total_optuna:.1f}s ({t_total_optuna/60:.1f} min, {t_total_optuna/3600:.2f} h)')
print(f'   Desglose: L1={trial_times["L1"]/60:.1f} min | L2={trial_times["L2"]/60:.1f} min | L3={trial_times["L3"]/60:.1f} min')
print(f"{'='*60}")


Optimizando TARGET: L1 — Ingreso hospitalario


  0%|          | 0/30 [00:00<?, ?it/s]


L1 — Mejor AUROC (Optuna, max_epochs=10, 30 trials): 0.8052
   Trials completos: 30 | pruned: 0
   Tiempo: 10089.9s (168.2 min) | Media por trial: 336.3s
   Mejores parámetros: {'units': 64, 'n_layers': 3, 'dropout': 0.005050948672369694, 'fc_hidden': 64, 'lr': 0.0007935097591921035, 'batch_size': 1024}

Optimizando TARGET: L2 — Resultado crítico


  0%|          | 0/30 [00:00<?, ?it/s]


L2 — Mejor AUROC (Optuna, max_epochs=10, 30 trials): 0.8881
   Trials completos: 30 | pruned: 0
   Tiempo: 10482.3s (174.7 min) | Media por trial: 349.4s
   Mejores parámetros: {'units': 64, 'n_layers': 1, 'dropout': 0.10831729227550749, 'fc_hidden': 64, 'lr': 0.0015957360356313554, 'batch_size': 1024}

Optimizando TARGET: L3 — Intervención crítica


  0%|          | 0/30 [00:00<?, ?it/s]


L3 — Mejor AUROC (Optuna, max_epochs=10, 30 trials): 0.9003
   Trials completos: 30 | pruned: 0
   Tiempo: 28711.5s (478.5 min) | Media por trial: 957.1s
   Mejores parámetros: {'units': 256, 'n_layers': 3, 'dropout': 0.10718028500106341, 'fc_hidden': 16, 'lr': 0.0008252372316470848, 'batch_size': 1024}

Búsqueda completada para todos los targets.
   Tiempo TOTAL Optuna (3 targets x 30 trials): 49283.8s (821.4 min, 13.69 h)
   Desglose: L1=168.2 min | L2=174.7 min | L3=478.5 min


## 6. Reentrenamiento final

Se reentrena con los mejores hiperparámetros y `max_epochs=30, patience=7` para obtener el rendimiento real del modelo optimizado, sin las restricciones de velocidad de los trials.

In [6]:
final_models  = {}
final_results = {}

for target in TARGETS:
    y_tr = df_train[target].values.astype(np.float32)
    y_v  = df_val[target].values.astype(np.float32)
    bp   = best_params[target]
    pos_weight = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))

    print(f"\n{'='*60}\nREENTRENANDO: {target} — {TARGET_NAMES[target]}\n{'='*60}")

    model = build_model(
        n_vitals=X_seq_train.shape[2], n_static=X_static_train.shape[1],
        units=bp['units'], n_layers=bp['n_layers'], dropout=bp['dropout'],
        fc_hidden=bp['fc_hidden'], lr=bp['lr'],
    )
    model.fit(
        [X_seq_train, X_static_train], y_tr,
        validation_data=([X_seq_val, X_static_val], y_v),
        epochs=30, batch_size=bp['batch_size'],
        class_weight={0: 1.0, 1: pos_weight},
        callbacks=[
            EarlyStopping(monitor='val_auc', patience=7, mode='max',
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_auc', patience=3, factor=0.5,
                              mode='max', verbose=0),
        ],
        verbose=1,
    )

    y_pred = model.predict([X_seq_val, X_static_val], verbose=0).ravel()
    auroc = roc_auc_score(y_v, y_pred)
    auprc = average_precision_score(y_v, y_pred)
    brier = brier_score_loss(y_v, y_pred)
    print(f'\n  AUROC: {auroc:.4f} | AUPRC: {auprc:.4f} | Brier: {brier:.4f}')

    final_results[target] = {'AUROC': auroc, 'AUPRC': auprc, 'Brier': brier, 'Prev_val': float(y_v.mean())}
    final_models[target]  = model

print('\nReentrenamiento completado.')


REENTRENANDO: L1 — Ingreso hospitalario
Epoch 1/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 38s 111ms/step - auc: 0.7800 - loss: 0.6910 - val_auc: 0.7966 - val_loss: 0.5559 - learning_rate: 7.9351e-04
Epoch 2/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 29s 108ms/step - auc: 0.8024 - loss: 0.6600 - val_auc: 0.8015 - val_loss: 0.5503 - learning_rate: 7.9351e-04
Epoch 3/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 29s 108ms/step - auc: 0.8055 - loss: 0.6550 - val_auc: 0.8027 - val_loss: 0.5498 - learning_rate: 7.9351e-04
Epoch 4/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 30s 109ms/step - auc: 0.8061 - loss: 0.6537 - val_auc: 0.8033 - val_loss: 0.5490 - learning_rate: 7.9351e-04
Epoch 5/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 30s 109ms/step - auc: 0.8068 - loss: 0.6527 - val_auc: 0.8037 - val_loss: 0.5485 - learning_rate: 7.9351e-04
Epoch 6/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 30s 109ms/step - auc: 0.8073 - loss: 0.6519 - val_auc: 0.8040 - val_loss: 0.5478 - learning_rate: 7.9351e-04
Epoch 7/30
272/272 ━━━━━━━━━━━━━━━━━━━━ 29s 108ms/step - auc: 0.8

## 7. Resumen de resultados Optuna

Métricas del modelo LSTM optimizado en validación. Las comparativas entre arquitecturas (LogReg, CatBoost, LSTM, TabTransformer, NAM) se realizan en el notebook de evaluación con el test set.

In [7]:
rows = []
for target in TARGETS:
    o = final_results[target]
    rows.append({
        'Target':   target,
        'Nombre':   TARGET_NAMES[target],
        'AUROC':    o['AUROC'],
        'AUPRC':    o['AUPRC'],
        'Brier':    o['Brier'],
        'Prev_val': o['Prev_val'],
    })
df_summary = pd.DataFrame(rows).set_index('Target')
print('=== LSTM + Optuna — Resultados en validación ===')
print(df_summary.round(4).to_string())

=== LSTM + Optuna — Resultados en validación ===
                      Nombre   AUROC   AUPRC   Brier  Prev_val
Target                                                        
L1      Ingreso hospitalario  0.8058  0.7252  0.1813    0.3831
L2         Resultado crítico  0.8877  0.1458  0.1332    0.0147
L3      Intervención crítica  0.8985  0.1312  0.0932    0.0061


## 8. Guardado de artefactos

Se persisten los pesos del modelo (`.pt`), los hiperparámetros óptimos y la configuración de la arquitectura en `models/lstm/<timestamp>/` para su uso en la fase de evaluación y generación de predicciones.

In [8]:
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
MODELS_DIR = Path(f'../../models/lstm/{TIMESTAMP}')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Modelos Keras (.keras incluye arquitectura + pesos + config del optimizador)
for target, model in final_models.items():
    model.save(str(MODELS_DIR / f'lstm_{target}.keras'))

with open(MODELS_DIR / 'best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)
with open(MODELS_DIR / 'final_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

model_config = {
    'n_vitals':        int(X_seq_train.shape[2]),
    'n_static':        int(X_static_train.shape[1]),
    'max_len':         MAX_LEN,
    'static_features': STATIC_FEATURES,
    'cc_vocab':        {k: int(v) for k, v in cc_vocab.items()},
    'cc_top_n':        CC_TOP_N,
    'cc_oov_idx':      int(CC_OOV),
    'best_params':     best_params,
}
with open(MODELS_DIR / 'model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2, ensure_ascii=False)

with open(MODELS_DIR / 'preprocessors.pkl', 'wb') as f:
    pickle.dump({'imputer': imputer_s, 'scaler': scaler_s, 'ohe': ohe_s}, f)

print(f'Artefactos guardados en {MODELS_DIR}')

Artefactos guardados en ..\..\models\lstm\20260610_213549
